<a href="https://colab.research.google.com/github/aastha3399/ML_Lab_2026/blob/main/ML_EXP7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#Naive Bayes

import math
import numpy as np
import pandas as pd

# 1. Load the CSV file dynamically from the environment
file_path = "Sport car price.csv"
df = pd.read_csv(file_path)

# Generate text data out of the structural fields to perform text classification
df["text_feature"] = (
    df["Car Model"].astype(str) + " " + df["Year"].astype(str)
)  # [1]
df["target_label"] = df["Car Make"]  # [1]

print("Dataset Loaded and Features Created:")
print(df[["text_feature", "target_label"]].head(), "\n")


# 2. Text Preprocessing & Tokenization
def preprocess(text):
    text = text.lower()
    punctuation = '!"#$%&\'()*+,-./:;<=>?@[\\]^_`{|}~'
    cleaned_text = ""
    for char in text:
        if char not in punctuation:
            cleaned_text += char
    tokens = cleaned_text.split()
    return tokens


df["tokens"] = df["text_feature"].apply(preprocess)
documents = df["tokens"].tolist()
N = len(documents)

# 3. Feature Extraction (TF-IDF built using basic loop structures)
df_counts = {}
for doc in documents:
    unique_words_in_doc = set(doc)
    for word in unique_words_in_doc:
        if word in df_counts:
            df_counts[word] += 1
        else:
            df_counts[word] = 1

vocab = sorted(list(df_counts.keys()))


def get_tfidf(doc, vocab, df_counts, N):
    tf_counts = {}
    for word in doc:
        tf_counts[word] = tf_counts.get(word, 0) + 1

    doc_len = len(doc) if len(doc) > 0 else 1
    vector = []
    for word in vocab:
        tf = tf_counts.get(word, 0) / doc_len
        word_df = df_counts.get(word, 0)
        idf = math.log((1 + N) / (1 + word_df)) + 1
        vector.append(tf * idf)
    return vector


# Convert row items to text feature matrix arrays
X = np.array([get_tfidf(doc, vocab, df_counts, N) for doc in documents])
y = df["target_label"].values

# 4. Split Data (Using an 80/20 train/test ratio split)
split_idx = int(0.8 * N)
X_train, X_test = X[:split_idx], X[split_idx:]
y_train, y_test = y[:split_idx], y[split_idx:]

# 5. Train Naive Bayes Classifier (Gaussian adjustment for continuous TF-IDF scores)
classes = []
for label in y_train:
    if label not in classes:
        classes.append(label)

class_priors = {}
class_feature_means = {}
class_feature_vars = {}

for c in classes:
    X_c = np.array([X_train[i] for i in range(len(y_train)) if y_train[i] == c])

    class_priors[c] = len(X_c) / len(X_train)
    # Apply variance smoothing protection depending on class array shape
    if len(X_c) > 1:
        class_feature_means[c] = np.mean(X_c, axis=0)
        class_feature_vars[c] = np.var(X_c, axis=0) + 1e-4
    else:
        class_feature_means[c] = X_c
        class_feature_vars[c] = np.zeros(len(vocab)) + 1e-4


# 6. Make Predictions
def predict_row(row):
    best_class = None
    best_prob = -float("inf")

    for c in classes:
        prior = math.log(class_priors[c])
        mean = class_feature_means[c]
        var = class_feature_vars[c]

        log_lik = -0.5 * np.sum(
            np.log(2 * math.pi * var) + ((row - mean) ** 2) / var
        )
        posterior = prior + log_lik

        if posterior > best_prob:
            best_prob = posterior
            best_class = c
    return best_class


y_pred = np.array([predict_row(row) for row in X_test])

# 7. Evaluate Model
accuracy = np.mean(y_pred == y_test)
print(f"Test Accuracy: {accuracy * 100:.2f}%\n")

print("Classification Report:")
for c in classes:
    tp, fp, fn = 0, 0, 0
    for i in range(len(y_test)):
        if y_pred[i] == c and y_test[i] == c:
            tp += 1
        elif y_pred[i] == c and y_test[i] != c:
            fp += 1
        elif y_pred[i] != c and y_test[i] == c:
            fn += 1

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = (
        (2 * precision * recall) / (precision + recall)
        if (precision + recall) > 0
        else 0
    )

    print(
        f"Class '{c}': Precision={precision:.2f}, Recall={recall:.2f}, F1-Score={f1:.2f}"
    )


Dataset Loaded and Features Created:
   text_feature target_label
0      911 2022      Porsche
1  Huracan 2021  Lamborghini
2  488 GTB 2022      Ferrari
3       R8 2022         Audi
4     720S 2021      McLaren 

Test Accuracy: 90.10%

Classification Report:
Class 'Porsche': Precision=1.00, Recall=1.00, F1-Score=1.00
Class 'Lamborghini': Precision=1.00, Recall=1.00, F1-Score=1.00
Class 'Ferrari': Precision=1.00, Recall=1.00, F1-Score=1.00
Class 'Audi': Precision=1.00, Recall=1.00, F1-Score=1.00
Class 'McLaren': Precision=1.00, Recall=0.60, F1-Score=0.75
Class 'BMW': Precision=1.00, Recall=1.00, F1-Score=1.00
Class 'Mercedes-Benz': Precision=1.00, Recall=0.15, F1-Score=0.27
Class 'Chevrolet': Precision=1.00, Recall=1.00, F1-Score=1.00
Class 'Ford': Precision=0.90, Recall=0.82, F1-Score=0.86
Class 'Nissan': Precision=1.00, Recall=1.00, F1-Score=1.00
Class 'Aston Martin': Precision=1.00, Recall=1.00, F1-Score=1.00
Class 'Bugatti': Precision=1.00, Recall=1.00, F1-Score=1.00
Class 'Dodge': 